### **The Inventory Baseline**

This inventory baseline establishes the ***Ideal State*** of the supply chain network. 

The goal is to generate a baseline inventory before the realism (chaos) is introduced to the data. The inventory tracks inventory movement across 10 nodes for the month of march 2026. 

#### **Key Logic and Constraints**
- ***Temporal resolution*** : The daily snapshots are taken at midnight representing the EOD inventory. 

- ***Modelled dynamics*** : 
1. Seasonality : The sales logic accounts for spikes at month start and during weekends. 
2. Friction metrics : Each record tracks processing_lag_hours and return reason (rto_reason) to provide an audit trail for returns and delivery delays. 

- ***The data contract*** : Each product-node pair is evaluated daily to track the relationship between stock on hand and fulfilment velocity. 

#### ***Schema example :***
___
| timestamp | node_id | product_id | stock_on_hand | units_sold| units_rto | rto_reason | processing_lag_hours |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 2026-03-01 | NODE_001 | PROD_001 | 9184 | 130| 0 | None | 3.757 |
___



In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [2]:
### Load the data
df_prod = pd.read_csv('01_Test_products.csv')
df_node = pd.read_csv('01_Test_nodes.csv')

In [3]:
records = []
start_date = datetime(2026,3,1) 
for day in range(31):
    curr_date = start_date + timedelta(days=day)
    ### Month-start and weekends sale hikes
    is_weekend = curr_date.weekday() in [5,6]
    is_month_start = curr_date.day <=3

    for _, node in df_node.iterrows():
        for _, prod in df_prod.iterrows():
            ### 1. Baseline Logic: Hubs will have large stock and Darkstores are going to have smaller stock
            if node['type'] == 'Hub':
                stock = np.random.randint(5000,10000)
                base_sales = np.random.randint(100,500)
            else:
                stock = np.random.randint(20,150)
                base_sales = np.random.randint(5,30)

            ### 2. factor-in the sales hike during weekends and month start
            if is_weekend:
                sales = int(base_sales * 1.3)
            elif is_month_start:
                sales = int(base_sales * 1.4)
            else:
                sales = base_sales
            ### 3. RTO logic
            units_rto = np.random.randint(0,sales//5)
            if units_rto > 0:
                rto_reason = np.random.choice(['cancelled', 'customer unavailable', 'incorrect address'])
            else:
                rto_reason = 'None'
            ### append the record
            records.append({
                'timestamp': curr_date,
                'node_id': node['node_id'],
                'product_id': prod['product_ID'],
                'stock_on_hand': max(0, stock - sales),
                'units_sold': sales,
                # 'is_rto': False, # Returned order
                'units_rto': units_rto, 
                'rto_reason': rto_reason,
                'processing_lag_hours': np.random.uniform(0.5, 4.0) # The "Phantom Inventory" seed
            })

df_inventory = pd.DataFrame(records)
df_inventory.to_csv('02_Test_Inventory_baseline.csv', index = False)
print("Success: Baseline 31 days inventory state generated")

Success: Baseline 31 days inventory state generated
